# HW4: Approximate Bayesian Computation, Variational Inference, & Simulation-Based Inference

## Problem 1
We want to measure the volumetric rate of Type Ia supernovae, $\lambda$ (the expected number of SNe occurring within your survey's volume and duration), from a magnitude-limited survey.

Let's say simulating this observation is straightforward: the number of SNe that occur is Poisson with mean $\lambda$.
Each SN has a true peak apparent magnitude $m_{\rm true}\sim\mathcal{U}(18,24)$ and it's observed with intrinsic scatter $m_{\rm obs} = m_{\rm true} + \mathcal{N}(0,\sigma_M)$, $\sigma_M=0.1$ mag. 
The SN is only detected with probability given by the selection function: 
$$c(m) = \frac{1}{1+\exp[(m-m_{50})/w]}, \qquad m_{50}=23.0,\ w=0.3.$$
Deriving the analytic likelihood for the observed data given $\lambda$ is intractable, but simulating the whole process is trivial.

**(a)** Implement `simulate(lam)`, which returns the array of observed (detected) magnitudes for expected rate $\lambda$. Using $\lambda_{\rm true}=250$, generate one mock "observed" dataset and record its summary statistics: $N_{\rm obs}$ (number detected) and $\bar m_{\rm obs}$ (mean of detected magnitudes).

*Hint: `np.random.poisson`, `np.random.uniform`.*

**(b)** Choose a prior $\lambda\sim\mathcal{U}(50,500)$, implement `sample_prior()`, and draw $\gtrsim 2\times10^4$ pairs $(\lambda', \mathcal D')$ by repeatedly sampling the prior and simulating. Plot $\lambda'$ vs. $N'_{\rm obs}$ and examine $P(\lambda, N_{\rm obs})$

**(c)** Implement ABC using $N_{\rm obs}$ alone as your summary statistic with the criterion $|N'_{\rm obs}-N_{\rm obs}|<\epsilon$. Examine the posterior of $\lambda$ for a few choices of $\epsilon$ (e.g. 30, 15, 5). What's the accuracy/efficiency trade-off as $\epsilon\to0$?

**(d)** Now also include $\bar m_{\rm obs}$ in your distance metric. Does adding this second summary statistic tighten your posterior on $\lambda$? What does $\bar m_{\rm obs}$ actually depends on? How does this explain your findings?

*Hint: e.g. normalized summary statistics with euclidean distance*

**(e)** Why is the likelihood for $\mathcal D$ given $\lambda$ intractable here, and why couldn't you just plug this problem into MCMC?

## Problem 2
Many of the next generation galaxy surveys are emission-line surveys that rely on a single emission line at some observed wavelength $\lambda_{\rm obs}$ to measure galaxy redshifts: $z = \lambda_{\rm obs}/\lambda_{\rm rest} - 1$. 
One major issue is line confusion. 
A survey searching for high-redshift Lyman-$\alpha$ emitters (LAEs) look for emission lines at $\lambda_{\rm rest}=1215.67A$ but they will also pick up [OII] emitters at lower redshifts ($\lambda_{\rm rest}=3727A$).

Each line implies a different redshift and the measurement uncertainty in $\lambda$ ($\sigma_\lambda$) propagates to a different $\sigma_z = \sigma_\lambda/\lambda_{\rm rest}$ for each line. 
Treating either line as equally likely we can write down the posterior
$$p^*(z) = \tfrac12\,\mathcal N(z; z_{\rm Ly\alpha}, \sigma_{\rm Ly\alpha}) + \tfrac12\,\mathcal N(z; z_{\rm OII}, \sigma_{\rm OII})$$

**(a)** Using $\lambda_{\rm obs}=4000A$ and uncertainty $\sigma_\lambda=5A$, compute $z_{\rm Ly\alpha}, \sigma_{\rm Ly\alpha}, z_{\rm OII}, \sigma_{\rm OII}$. 
Implement `log_pstar(z)` for the mixture above in `torch`, evaluate it on a grid, and plot $p^*(z)$.

*Hint: `torch.distributions.Normal(...).log_prob(...)` and `torch.logsumexp`*

**(b)** Implement variational inference : fit a single Gaussian surrogate $q_\phi(z) = \mathcal N(\mu,\sigma)$ by minimizing $D_{\rm KL}(q_\phi\parallel p^*)$ via the reparameterization trick and the Adam optimizer. 
Overplot $q_\phi$ on your $p^*(z)$ from (a). 
Which redshift solution (if any) does it lock onto?

**(c)** Now fit a 2-component Gaussian-mixture surrogate $q_\phi(z)$. 
Compare to (b). 
Does it recover both $z_{\rm Ly\alpha}$ and $z_{\rm OII}$? 
Report the mean/std of each component against the true values from (a).

**(d)** Suppose an automated pipeline used a single-Gaussian VI fit like (b) to assign redshifts across a whole survey. 
What would go wrong, and how could this bias scientific analyses?
In practice, what additional information could you use to break this degeneracy?

## Problem 3
In Problem 1, reducing $\epsilon$ to tighten your ABC posterior required throwing away a larger and larger fraction of your simulations. 
This is a serious problem if each simulation were computationally expensive, e.g., a cosmological N-body run.

Let's instead fit a cheap surrogate to the *joint* distribution $p(\lambda, \mathcal D)$ using variational inference, then run ABC on samples drawn from that surrogate instead of the real simulator.

**(a)** Using your Problem 1 (b) samples $\{(\lambda', N'_{\rm obs})\}$, fit a Gaussian surrogate $q_\phi(\lambda,\mathcal D)$ by minimizing the forward KL divergence $D_{\rm KL}(p\parallel q_\phi) \approx -\langle\log q_\phi(\lambda',\mathcal D')\rangle$ over your samples.

*Hint: use Cholesky factorization to keep the covariance positive-definite.*

**(b)** Draw $\gtrsim 10^6$ cheap samples from $q_\phi$ and run ABC on these with a much smaller $\epsilon$ than you could afford in Problem 1(c). 
Compare the resulting posterior on $\lambda$ to your Problem 1(c) result. 
How many simulator calls did your Problem 1(b) training set use versus the number of effective ABC samples you now have access to?

**(c)** Under what condition would this two-step (VI surrogate + ABC) approach silently fail?

## Problem 4
In HW3 Problem 4, you ran `emcee` on the full 12-parameter `provabgs` model to fit the SDSS spectrum `spec-0279-51608-0017.fits` — thousands of expensive model evaluations for one galaxy. 
Let's see how well a much cheaper VI/ABC approach gets us on the single parameter you care about most: `logmstar`.

To make this tractable, we'll fix every other parameter at your HW3 posterior median (or reasonable defaults if you didn't reach HW3 Problem 4). 
We'll also compress the ~4000-pixel spectrum down to one summary statistic: the total observed flux, $S = \sum_i f_i$ over the unmasked pixels.

In [ ]:
# same setup as HW3 Problem 4
from astropy.io import fits
import matplotlib.pyplot as plt

f = fits.open('spec-0279-51608-0017.fits')
wave = 10**f[1].data['loglam']
mask = f[1].data['and_mask'].astype(bool) # mask
flux = f[1].data['flux']
ivar = f[1].data['ivar'] # inverse variance
zred = f[2].data['Z'] # redshift

plt.figure(figsize=(10,4))
plt.plot(wave[~mask], flux[~mask], label='Flux')
plt.plot(wave[~mask], ivar[~mask]**-0.5, label='Variance')
plt.legend(loc='upper right')
plt.ylim(0, None)

print(f'galaxy redshift = {zred}')

In [ ]:
from provabgs import infer as Infer
from provabgs import models as Models

# stellar population synthesis model
model = Models.NMF(burst=True, emulator=True)

# default prior (same as HW3)
prior = Infer.load_priors([
    Infer.UniformPrior(7., 12.5, label='sed'),
    Infer.FlatDirichletPrior(4, label='sed'),   # flat dirichilet priors
    Infer.UniformPrior(0., 1., label='sed'), # burst fraction
    Infer.UniformPrior(1e-2, 13.27, label='sed'), # tburst
    Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'), # log uniform priors on ZH coeff
    Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'), # log uniform priors on ZH coeff
    Infer.UniformPrior(0., 3., label='sed'),        # uniform priors on dust1
    Infer.UniformPrior(0., 3., label='sed'),        # uniform priors on dust2
    Infer.UniformPrior(-2., 1., label='sed')    # uniform priors on dust_index
])

**(a)** Fix every parameter except `logmstar` at your HW3 posterior means (or reasonable defaults, e.g. the prior midpoints, if you didn't complete HW3 Problem 4). 
Implement `simulate(logmstar)` by 
1. build the full parameter vector
2. call `model.sed(theta, zred=zred, wave=wave)`
3. add noise drawn from `ivar` at each unmasked pixel
4. return $S=\sum f_i$ over unmasked pixels
  
Also compute $S_{\rm obs}$ from the real data.

**(b)** Draw `logmstar` from its prior (same as in HW3) and generate $\gtrsim 5000$ pairs $(\theta', S')$ using `simulate`. 
This is your full simulation budget! Plot the joint distribution of $\theta$ and $S$

**(c)** Run ABC on these draws to get a posterior on `logmstar`. 
In parallel, treat $p(S\,|\,{\tt logmstar})$ as approximately Gaussian and fit a Gaussian surrogate $q_\phi({\tt logmstar})$ via standard VI --- an analytic Gaussian likelihood for $S$ centered on the simulated mean/scatter at each `logmstar`. 
Is this a reasonable approximation? 
Compare the ABC and VI posteriors to each other.

**(d)** Compare both posteriors here to your `emcee` marginal posterior on `logmstar`. Do they agree? 

**(e)** In this problem, we collapsed a 12-dimensional model and a ~4000-pixel dataset down to 1 free parameter and 1 summary statistic. Discuss what would need to change to do full SBI on all 12 `provabgs` parameters using the full spectrum? 